In [0]:
/*
===============================================================================
DDL Script: Create Gold Views
===============================================================================
Script Purpose:
    This script creates views for the Gold layer. 
    The Gold layer represents the final dimension and fact tables (Star Schema)

    Each view performs transformations and combines data from the Silver layer 
    to produce a clean, enriched, and business-ready dataset.

Usage:
    - These views can be queried directly for analytics and reporting.
===============================================================================
*/

-- =============================================================================
-- Create Dimension: gold.dim_customers
-- =============================================================================
DROP VIEW IF EXISTS gold.dim_customers;

CREATE OR REPLACE VIEW gold.dim_customers AS
SELECT
    customer_id,
    customer_unique_id,
    customer_city,
    customer_state,
    customer_zip_code_prefix
FROM silver.customers;

-- =============================================================================
-- Create Dimension: gold.dim_sellers
-- =============================================================================
DROP VIEW IF EXISTS gold.dim_sellers;

CREATE OR REPLACE VIEW gold.dim_sellers AS
SELECT 
    seller_id,
    seller_city,
    seller_state,
    seller_zip_code_prefix
FROM silver.sellers;

-- =============================================================================
-- Create Dimension: gold.dim_products
-- =============================================================================
DROP VIEW IF EXISTS gold.dim_products;

CREATE OR REPLACE VIEW gold.dim_products AS
SELECT 
    p.product_id,
    p.product_category_name AS category_pt,
    COALESCE(t.product_category_name_english, 'unknown') AS category_en,
    p.product_weight_g,
    p.product_length_cm,
    p.product_height_cm,
    p.product_width_cm
FROM silver.products p
LEFT JOIN silver.product_category_name_translation t 
  ON p.product_category_name = t.product_category_name;

-- =============================================================================
-- Create Dimension: gold.dim_date
-- =============================================================================
DROP VIEW IF EXISTS gold.dim_date;

CREATE OR REPLACE VIEW gold.dim_date AS
SELECT
    date_format(d, 'yyyyMMdd') AS date_id,
    d AS full_date,
    year(d) AS year,
    month(d) AS month,
    day(d) AS day,
    date_format(d, 'EEEE') AS day_name,
    quarter(d) AS quarter
FROM (SELECT explode(sequence(to_date('2010-01-01'), to_date('2020-12-31'), interval 1 day)) AS d);

-- =============================================================================
-- Create Fact Table: gold.fact_order_items
-- =============================================================================
DROP VIEW IF EXISTS gold.fact_order_items;

CREATE OR REPLACE VIEW gold.fact_order_items AS
SELECT 
    oi.order_id,
    oi.order_item_id,
    oi.product_id,
    oi.seller_id,
    o.customer_id,
    oi.price,
    oi.freight_value,
    oi.total_item_value,
    oi.shipping_ratio,
    date_format(o.order_purchase_timestamp, 'yyyyMMdd') AS purchase_date_id
FROM silver.order_items oi
JOIN silver.orders o ON oi.order_id = o.order_id;

-- =============================================================================
-- Create Fact Table: gold.fact_orders
-- =============================================================================
DROP VIEW IF EXISTS gold.fact_orders;

CREATE OR REPLACE VIEW gold.fact_orders AS
SELECT 
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp,
    order_delivered_customer_date,
    -- Logistics KPIs
    datediff(order_delivered_customer_date, order_purchase_timestamp) AS actual_delivery_days,
    datediff(order_estimated_delivery_date, order_purchase_timestamp) AS estimated_delivery_days,
    CASE WHEN order_delivered_customer_date > order_estimated_delivery_date THEN 1 ELSE 0 END AS is_late
FROM silver.orders;

-- =============================================================================
-- Create Fact Table: gold.fact_payments
-- =============================================================================
DROP VIEW IF EXISTS gold.fact_payments;

CREATE OR REPLACE VIEW gold.fact_payments AS
SELECT 
    order_id,
    payment_type,
    payment_installments,
    payment_value
FROM silver.order_payments;

-- =============================================================================
-- Create Fact Table: gold.fact_reviews
-- =============================================================================
DROP VIEW IF EXISTS gold.fact_reviews;

CREATE OR REPLACE VIEW gold.fact_reviews AS
SELECT 
    review_id,
    order_id,
    review_score,
    date_format(review_creation_date, 'yyyyMMdd') AS review_creation_date,
    date_format(review_answer_timestamp, 'yyyyMMdd') AS review_answer_date
FROM silver.order_reviews;